# **Misinformation Correction with MUSE Framework**

This notebook demonstrates the use of the MUSE (Multimodal Search-and-Summarize) framework for correcting misinformation in Spanish news articles. The system uses Google Gemini LLM to generate queries, retrieve evidence from web sources, and generate fact-checked corrections.

## **1. Setup and Imports**

Import necessary libraries and load API keys for Gemini and Google Custom Search.

In [ ]:
import sys
import textwrap
import json
from google import genai
import random
sys.path.append('/Users/vasylkorzavatykh/Documents/Studies/LLM-Misinformation-Correction')

from model.main import correct_article, download_news_articles

api_keys = json.load(open('../model/data/api_keys.json'))

### **1.1. Helper Functions**

We define utility functions for text translation and formatting:
- `translate_to_english()`: Translates Spanish text to English using Gemini API
- `pretty_print()`: Formats text output for better readability (150 character width)

In [ ]:
def translate_to_english(text):
    # Read the Gemini API key from api_keys.json
    gemini_api_key = api_keys['gemini']
    
    # Create Gemini client
    client = genai.Client(api_key=gemini_api_key)

    # Use Gemini to translate from Spanish to English
    prompt = (
        "Translate the following text from Spanish to English. "
        "Just return the English translation and nothing else:\n\n"
        f"{text}"
    )
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )
    return response.text.strip()

def pretty_print(text):
    print(textwrap.fill(text, width=150))

## **2. Exemplary Usage**

This section demonstrates how to use the MUSE framework to fact-check news articles.

### **2.1. Load the Dataset**

Download and load the Spanish Fake News Corpus containing 572 articles (286 fake, 286 true news).

In [3]:
data = download_news_articles()
data

,ID,CATEGORY,TOPICS,SOURCE,HEADLINE,TEXT,LINK
0,1,1,Covid-19,El Economista,Covid-19: mentiras que matan,El control de la Covid-19 no es sólo un tema d...,https://www.eleconomista.com.mx/opinion/Covid-...
1,2,0,Política,El matinal,El Gobierno podrá acceder a las IPs de los móv...,El Gobierno de Pedro Sánchez y Pablo Iglesias ...,https://www.elmatinal.com/espana-ultima-hora/e...
2,3,1,Política,El País,La comunidad musulmana catalana denuncia a Vox...,Las tres federaciones que agrupan al 90% de la...,https://elpais.com/espana/elecciones-catalanas...
3,4,0,Política,AFPFactual,,Se han dado a conocer los datos electorales pr...,https://perma.cc/GYE6-SPMB
4,5,1,Sociedad,La Republica,El censo poblacional 2018 tendrá un costo de $...,La primera fase del censo será virtual y solo ...,https://www.larepublica.co/economia/el-censo-p...
...,...,...,...,...,...,...,...
567,568,1,Covid-19,El Financiero,Encuentran nueva variante de COVID en México: ...,El Instituto de Diagnóstico y Referencia Epide...,https://www.elfinanciero.com.mx/salud/encuentr...
568,569,0,Sociedad,diariogol,El móvil de más de 60.000 euros de la princesa...,La hija del rey Felipe y de la reina Letizia y...,https://www.diariogol.com/gossip/el-movil-de-m...
569,570,0,Política,AFPFactual,,"Evidentemente, Barak Obama ha sido arrestado e...","Perma | Obama, Biden y la directora de la CIA,..."
570,571,1,Covid-19,Redacción Médica,Covid: las vacunas puestas en Espa?a no alcanz...,El Ministerio de Sanidad ha actualizado los da...,https://www.redaccionmedica.com/secciones/sani...


### **2.2. Select an Article for Fact-Checking**

Choose an article by ID (1-572) to analyze. The output shows the article's category (true/fake) and its full text content.

In [ ]:
ID_to_test = 1 # Possibilities to choose: 1 - 572
news_to_test = data[data['ID'] == str(ID_to_test)]
print(f"NEWS NUMBER: {ID_to_test}, true (in reality)?: {'yes' if news_to_test.CATEGORY.values[0] else 'no'}")
print()
pretty_print(news_to_test.TEXT.values[0])

NEWS NUMBER: 1, fake (in reality)?: yes

El control de la Covid-19 no es sólo un tema de médicos y el resto del personal sanitario y científico. Por desgracia o por fortuna, es un asunto
esencialmente político que se decide por hombres y mujeres que se dedican a la política. De las creencias y opiniones de estos últimos, depende el
éxito o el fracaso de las acciones que se implementen.    Los éxitos en la toma de decisiones salvan vidas y naciones; obviamente, los errores matan y
más si están acompa?ados de mentiras y medias verdades. En este sentido, durante el pasado Pulso de la Salud (9 de febrero) el presidente López rompió
un récord: en los primeros diez minutos había dicho tres mentiras graves o medias verdades, que también son mentiras. El problema con esto es que las
mentiras matan.    En esa ocasión, López Obrador dijo que afortunadamente se estaban reduciendo los contagios en todo el país. Poco después, el
subsecretario López-Gatell fue por este camino y complementó con la 

### **2.3. Translate Article to English (Optional)**

For better readability, we can translate the Spanish article to English using the Gemini translation function. This step is optional as the correction pipeline can work with Spanish text directly.

In [25]:
# Sample usage: translate the text of the first article (row with ID 1)
spanish_text = data[data['ID'] == "1"].TEXT.values[0]
translated_text = translate_to_english(spanish_text)

print("Translated article:\n")
pretty_print(translated_text)

Translated article:

Controlling Covid-19 is not just a matter for doctors and other health and scientific personnel. Unfortunately or fortunately, it is an essentially
political issue decided by men and women dedicated to politics. The success or failure of the actions implemented depends on the beliefs and opinions
of the latter.  Successful decision-making saves lives and nations; obviously, mistakes kill, especially if accompanied by lies and half-truths. In
this regard, during the last 'Pulse of Health' (February 9), President López broke a record: in the first ten minutes, he had told three serious lies
or half-truths, which are also lies. The problem with this is that lies kill.  On that occasion, López Obrador said that “fortunately” infections were
decreasing nationwide. Shortly after, Undersecretary López-Gatell followed suit and complemented this with the thesis that there had been a downward
trend in the last two weeks. This is a half-truth. Since early last December, sever

### **2.4. Generate Fact-Checked Correction**

Run the complete MUSE pipeline to fact-check the selected article. This process:
1. Generates search queries from the article content
2. Searches web pages using Google Custom Search (High/Medium/Low priority domains)
3. Crawls and filters retrieved articles based on similarity
4. Extracts explicit and implicit refutation evidence
5. Generates a fact-checked correction with source citations

**Note**: This process may take several minutes as it involves web crawling and multiple API calls.

In [28]:
# Generating correction of the news
correction = correct_article(ID_to_test)

Start correcting news
Generate queries from misinformation...
Search web pages with High priority...
	(sent 1 request to google programmable search)
	(sent 1 request to google programmable search)
Start selecting retrieved web pages...
Extract retrieved web page content...


8it [00:13,  1.69s/it]


	(no article found)
Search web pages with Medium priority...
	(sent 1 request to google programmable search)
	(sent 1 request to google programmable search)
Start selecting retrieved web pages...
Extract retrieved web page content...


8it [00:00, 13963.56it/s]

	(no article found)
Search web pages with Low priority...


...

14it [00:28,  3.39s/it]

	(fail to get https://www.frontiersin.org/journals/public-health/articles/10.3389/fpubh.2022.932010/pdf)


28it [00:36,  1.31s/it]


Compute the similarity between web page and misinformation content...


Extract evidence from selected web pages...
	(sent 1 request to llm)
	(sent 1 request to llm)
	(sent 1 request to llm)
	(stop with sufficient refutations)
Generate correction with 4 refutations...


### **2.5. Display the Generated Correction**

View the fact-checked correction generated by the MUSE framework. The correction explains where and why the news is misinformed or potentially misleading, with supporting evidence and source URLs.

In [ ]:
print("Correction of the article:")
print()
pretty_print(correction)

Correction of the article:

This news is potentially misleading if it suggests that COVID-19 impacted all social strata equally or that healthcare workers in Mexico received
sufficient and high-quality protection from their employers. Facts indicate a high correlation between lower social strata and infection risk due to
economic necessity and inadequate social protection programs, especially in Mexico City. Healthcare workers frequently received deficient employer-
provided personal protective equipment (PPE), despite mandatory employer obligations, leading to self-procurement of better gear and high rates of
infection, hospitalization, and death, particularly in Mexico City. Moreover, analyses relying solely on municipal-level data may be erroneous as they
often fail to account for urban segregation and heterogeneity, potentially misrepresenting the actual distribution of cases among poorer
neighborhoods.  https://blogs.lse.ac.uk/latamcaribbean/2021/03/02/la-pandemia-contra-los-pobre

**Note**: Even though this article is categorized as true news in the dataset, the correction correctly identifies it as potentially misleading. This may be due to emerging data or new information that became available after the dataset was created, demonstrating the system's ability to provide nuanced analysis beyond simple binary classification.

## **3. Experimental Evaluation**

This section evaluates the MUSE framework's performance on a random sample of articles from the dataset.

### **3.1. Random Article Selection**

Select a random sample of articles using a fixed seed (42) for reproducibility. This ensures consistent evaluation results across runs.

In [16]:
SEED = 42

ids = data['ID'].tolist()
random.shuffle(ids, random.seed(SEED))
ids = ids[:10]

print(f"IDs: {ids}")
ids_types = ['yes' if data[data['ID'] == str(id)].CATEGORY.values[0] == "1" else 'no' for id in ids]
print(f"True: {ids_types}")


IDs: ['372', '281', '93', '568', '133', '466', '270', '222', '75', '384']
True: ['yes', 'no', 'yes', 'yes', 'yes', 'no', 'no', 'no', 'no', 'yes']


### **3.2. Manual Evaluation Subset**

For manual evaluation, we select a subset of 5 articles from the randomly sampled set. These articles are manually reviewed to verify the accuracy of the generated corrections.

**Note**: The commented line shows the IDs of articles used for the final evaluation.

In [ ]:
# ids = ['466', '270', '222', '75', '384']

### **3.3. Generate Corrections for Selected Articles**

Process each selected article through the MUSE pipeline to generate fact-checked corrections. For each article:
- Display the article ID and its ground truth category (fake/true)
- Run the complete correction pipeline
- Display the generated correction with evidence and source citations

**Note**: This process is time-intensive and may hit API rate limits. The system includes retry logic and caching to handle these issues.

In [ ]:
for id in ids:
    print(f"ID: {id} - True: {'yes' if data[data['ID'] == str(id)].CATEGORY.values[0] == '1' else 'no'}\n\n")
    correction = correct_article(id)
    print("\nCorrection of the article:\n")
    pretty_print(correction)
    print("\n\n--------------------------------\n\n")

### **3.4. Evaluation Results**

**Manual Evaluation Results:**
- **Sample size**: 5 articles (selected with seed=42)
- **Correct classifications**: 5/5
- **Accuracy**: 100%

The manual evaluation demonstrated that the MUSE framework correctly identified the category (fake/true) of all 5 randomly selected articles. Each correction accurately reflected whether the articles were fake (category 0) or true (category 1) according to the dataset labels, with appropriate evidence and reasoning provided.